In [6]:
#This shows how ICA separates mixed signals
import numpy as np
from sklearn.decomposition import FastICA

#Step 1: Create 2 Simple Independent Signals
S=np.array([[1,1],
           [2,-1],
           [3,1],
          [4,-1]])
#Step 2:Mix Them
A = np.array([[1,2],[3,4]])
X=S@A.T #Observed mixed data

#Step 3: Apply ICA
ica=FastICA(n_components=2,random_state=0)
U=ica.fit_transform(X)
print("Original Sources:\n",S)
print("Recovered Sources (approx):\n",U)

Original Sources:
 [[ 1  1]
 [ 2 -1]
 [ 3  1]
 [ 4 -1]]
Recovered Sources (approx):
 [[-1.  1.]
 [-1. -1.]
 [ 1.  1.]
 [ 1. -1.]]


In [11]:
#Implementation of autoencoders
#This code shows how an autoencoder compresses 4 input foeatures
#into 2 hidden features and then reconstructs the original

import numpy as np
from sklearn.neural_network import MLPRegressor

#Step 1: Create simple data (4 features)
X = np.array([[1, 2, 3, 4],
             [2, 3, 4, 5],
             [3, 4, 5, 6],
             [4, 5, 6, 7]],dtype=float)

#Step 2: Create Autoencoder (4 -> 2 -> 4)
autoencoder = MLPRegressor(hidden_layer_sizes=(2,),
                          activation='relu',
                          max_iter=5000,
                          random_state=0)

#Step 3: Train to reproduce input
autoencoder.fit(X,X)

#Step 4: Reconstruct
X_reconstructed = autoencoder.predict(X)

#Step 5: Print 2D hidden representation (the "4-->2" part)
W=autoencoder.coefs_[0] #(4x2)input->hidden weight
b=autoencoder.intercepts_[0] #(2,)hidden biases
hidden_2d=np.maximum(0,X@W+b) #ReLU: max(0, ...)

print("Original (4 features):\n",X)
print("\nHidden (2 features): [This is the 4->2 compression]:\n",np.round(hidden_2d,2))
print("\nReconstructed :\n",np.round(X_reconstructed,2))


Original (4 features):
 [[1. 2. 3. 4.]
 [2. 3. 4. 5.]
 [3. 4. 5. 6.]
 [4. 5. 6. 7.]]

Hidden (2 features): [This is the 4->2 compression]:
 [[4.35 1.56]
 [5.81 2.11]
 [7.27 2.66]
 [8.74 3.21]]

Reconstructed :
 [[2.05 2.76 3.4  3.99]
 [2.52 3.33 4.09 5.  ]
 [2.99 3.89 4.78 6.  ]
 [3.45 4.45 5.47 7.  ]]


In [18]:
#implementation of active learning
#the most uncertain data points for labelling
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

X,y = make_classification(n_samples=100,
                         n_features=2,
                         n_informative=2,
                         n_redundant=0,
                         random_state=0)
#start with 10 labled samples
labeled = np.arange(10)
unlabeled=np.arange(10,100)

model = LogisticRegression(solver="liblinear")

for i in range(5):
    model.fit(X[labeled],y[labeled])
    probs=model.predict_proba(X[unlabeled])[:,1]

    query = np.argmin(np.abs(probs-0.5))

    labeled=np.append(labeled,unlabeled[query])
    unlabeled=np.delete(unlabeled,query)

    print("Iteration:",i+1,"| Labeled size:",len(labeled))
    

Iteration: 1 | Labeled size: 11
Iteration: 2 | Labeled size: 12
Iteration: 3 | Labeled size: 13
Iteration: 4 | Labeled size: 14
Iteration: 5 | Labeled size: 15


In [20]:
## To Classify data by computing weighted combinations of radial basis function

# Simple 1D input data
X = np.array([[1],[2],[3],[4]])
y = np.array([0, 0, 1, 1])

#Choose 2 fixed RBF centers manually
centers = np.array([[1],[4]])
sigma = 1.0
#Gaussian RBF Function
def rbf(x,c):
    return np.exp(-np.linalg.norm(x-c)**2/(2*sigma**2))

#Build hidden layer output
H = np.array([[rbf(x,c) for c in centers]for x in X])

#Solve for weights using simple Linear equation
w=np.linalg.pinv(H)@y #Moore-Penrose inverse

#Predict
y_pred = H @ w
y_pred_class = (y_pred>0.5).astype(int)

print("Predicted:",y_pred_class)

Predicted: [0 0 1 1]
